In [1]:
import numpy as np

def hungarian_method_optimized(matrix):
    """
    Optimized Hungarian Method for assignment problems
    """
    def reduce_matrix(matrix):
        print("\n--- Step 1: Initial Matrix ---")
        print(matrix)
        
        # Step 1: Row reduction
        matrix -= np.min(matrix, axis=1).reshape(-1, 1)
        print("\n--- After Row Reduction ---")
        print(matrix)

        # Step 2: Column reduction
        matrix -= np.min(matrix, axis=0)
        print("\n--- After Column Reduction ---")
        print(matrix)
        
        return matrix

    def find_assignments(matrix, original_matrix):
        print("\n--- Finding Assignments ---")
        # Find the optimal assignment based on final zero positions
        assignment = []
        while np.sum(matrix == 0) > 0:
            for i in range(matrix.shape[0]):
                row_zeros = np.where(matrix[i, :] == 0)[0]
                if len(row_zeros) == 1:
                    j = row_zeros[0]
                    assignment.append((i, j))
                    matrix[i, :] = float('inf')  # Mark row as "used"
                    matrix[:, j] = float('inf')  # Mark column as "used"
                    break
        # Calculate the total cost using the original matrix
        total_cost = sum(original_matrix[i, j] for i, j in assignment)
        print(f"Assignments: {assignment}")
        print(f"Total Cost: {total_cost}\n")
        return assignment, total_cost

    # Convert the matrix to float to handle arithmetic operations
    original_matrix = matrix.astype(float)  # Save the original matrix
    matrix = matrix.astype(float)  # Work with a modifiable copy
    print("--- Initial Cost Matrix ---")
    print(matrix)
    
    # Reduce the matrix
    cost_matrix = reduce_matrix(matrix)
    
    # Find the optimal assignment
    assignment, total_cost = find_assignments(cost_matrix, original_matrix)
    
    print("--- Final Output ---")
    print(f"Optimal Assignment: {assignment}")
    print(f"Minimum Total Cost: {total_cost}")
    return assignment, total_cost

# Smaller Example Test Case
cost_matrix = np.array([
    [4, 1],
    [2, 3]
])

assignment, total_cost = hungarian_method_optimized(cost_matrix)


--- Initial Cost Matrix ---
[[4. 1.]
 [2. 3.]]

--- Step 1: Initial Matrix ---
[[4. 1.]
 [2. 3.]]

--- After Row Reduction ---
[[3. 0.]
 [0. 1.]]

--- After Column Reduction ---
[[3. 0.]
 [0. 1.]]

--- Finding Assignments ---
Assignments: [(0, 1), (1, 0)]
Total Cost: 3.0

--- Final Output ---
Optimal Assignment: [(0, 1), (1, 0)]
Minimum Total Cost: 3.0


In [2]:
import numpy as np

def add_dummy_rows_and_columns(matrix):
    """
    Add dummy rows or columns to make the matrix square, filling with zeros.
    """
    rows, cols = matrix.shape
    if rows < cols:  # Add dummy rows
        dummy_rows = np.zeros((cols - rows, cols))
        matrix = np.vstack([matrix, dummy_rows])
    elif cols < rows:  # Add dummy columns
        dummy_cols = np.zeros((rows, rows - cols))
        matrix = np.hstack([matrix, dummy_cols])
    return matrix

def reduce_matrix(matrix):
    """
    Perform row and column reductions on the cost matrix.
    """
    # Row reduction
    matrix -= np.min(matrix, axis=1).reshape(-1, 1)
    
    # Column reduction
    matrix -= np.min(matrix, axis=0)
    
    return matrix

def find_minimum_lines(matrix):
    """
    Cover all zeros in the matrix with the minimum number of lines.
    Returns the rows and columns that are covered.
    """
    covered_rows = set()
    covered_cols = set()
    zero_positions = np.argwhere(matrix == 0)

    # Iteratively cover zeros
    while zero_positions.size > 0:
        row_counts = np.sum(matrix == 0, axis=1)
        col_counts = np.sum(matrix == 0, axis=0)

        # Cover the row or column with the most zeros
        if np.max(row_counts) >= np.max(col_counts):
            row = np.argmax(row_counts)
            covered_rows.add(row)
            matrix[row, :] = np.inf  # Mark row as covered
        else:
            col = np.argmax(col_counts)
            covered_cols.add(col)
            matrix[:, col] = np.inf  # Mark column as covered

        # Update zero positions
        zero_positions = np.argwhere(matrix == 0)

    return covered_rows, covered_cols

def adjust_matrix(matrix, covered_rows, covered_cols):
    """
    Adjust the matrix when the number of covering lines is less than the size of the matrix.
    """
    # Find the smallest uncovered value
    uncovered_rows = np.setdiff1d(np.arange(matrix.shape[0]), list(covered_rows))
    uncovered_cols = np.setdiff1d(np.arange(matrix.shape[1]), list(covered_cols))
    
    if len(uncovered_rows) > 0 and len(uncovered_cols) > 0:
        uncovered = matrix[np.ix_(uncovered_rows, uncovered_cols)]
        min_value = np.min(uncovered)

        # Adjust uncovered and covered intersections
        matrix[np.ix_(uncovered_rows, uncovered_cols)] -= min_value
        matrix[np.ix_(list(covered_rows), list(covered_cols))] += min_value

    return matrix

def find_assignments(matrix, original_matrix):
    """
    Find the optimal assignment based on the final zero positions.
    Calculate the total cost using the original matrix.
    """
    assignment = []
    while np.sum(matrix == 0) > 0:
        for i in range(matrix.shape[0]):
            row_zeros = np.where(matrix[i, :] == 0)[0]
            if len(row_zeros) == 1:
                j = row_zeros[0]
                assignment.append((i, j))
                matrix[i, :] = float('inf')  # Mark row as used
                matrix[:, j] = float('inf')  # Mark column as used
                break
    # Calculate the total cost using the original matrix
    total_cost = sum(original_matrix[i, j] for i, j in assignment if i < original_matrix.shape[0] and j < original_matrix.shape[1])
    return assignment, total_cost

def hungarian_method(matrix):
    """
    Solve the assignment problem using the Hungarian Method.
    """
    print("\n--- Hungarian Method Started ---")
    
    # Save the original matrix (for calculating total cost)
    original_matrix = matrix.astype(float)
    
    # Ensure the matrix is square
    matrix = add_dummy_rows_and_columns(matrix.astype(float))
    print("\nInitial Matrix (after adding dummy variables if needed):")
    print(matrix)
    
    # Step 1: Reduce the matrix
    matrix = reduce_matrix(matrix)
    print("\nMatrix After Row and Column Reduction:")
    print(matrix)
    
    while True:
        # Step 2: Cover all zeros with minimum lines
        covered_rows, covered_cols = find_minimum_lines(matrix.copy())
        if len(covered_rows) + len(covered_cols) >= matrix.shape[0]:
            break
        # Step 3: Adjust the matrix
        matrix = adjust_matrix(matrix, covered_rows, covered_cols)
        print("\nMatrix After Adjustment:")
        print(matrix)
    
    # Step 4: Find the optimal assignment
    assignment, total_cost = find_assignments(matrix, original_matrix)
    print("\n--- Final Output ---")
    print(f"Optimal Assignment: {assignment}")
    print(f"Minimum Total Cost: {total_cost}")
    return assignment, total_cost

# Example Test Cases

# Test Case 1: A simple 3x3 cost matrix
cost_matrix_1 = np.array([
    [4, 2, 8],
    [2, 4, 7],
    [6, 6, 4]
])
print("\nTest Case 1: 3x3 Cost Matrix")
assignment_1, total_cost_1 = hungarian_method(cost_matrix_1)

# Test Case 2: A 2x3 non-square matrix
cost_matrix_2 = np.array([
    [3, 1, 2],
    [4, 3, 1]
])
print("\nTest Case 2: 2x3 Non-Square Matrix")
assignment_2, total_cost_2 = hungarian_method(cost_matrix_2)



Test Case 1: 3x3 Cost Matrix

--- Hungarian Method Started ---

Initial Matrix (after adding dummy variables if needed):
[[4. 2. 8.]
 [2. 4. 7.]
 [6. 6. 4.]]

Matrix After Row and Column Reduction:
[[2. 0. 6.]
 [0. 2. 5.]
 [2. 2. 0.]]

--- Final Output ---
Optimal Assignment: [(0, 1), (1, 0), (2, 2)]
Minimum Total Cost: 8.0

Test Case 2: 2x3 Non-Square Matrix

--- Hungarian Method Started ---

Initial Matrix (after adding dummy variables if needed):
[[3. 1. 2.]
 [4. 3. 1.]
 [0. 0. 0.]]

Matrix After Row and Column Reduction:
[[2. 0. 1.]
 [3. 2. 0.]
 [0. 0. 0.]]

--- Final Output ---
Optimal Assignment: [(0, 1), (1, 2), (2, 0)]
Minimum Total Cost: 2.0


In [3]:
import numpy as np

def add_dummy_rows_and_columns(matrix):
    """
    Add dummy rows or columns to make the matrix square, filling with zeros.
    """
    rows, cols = matrix.shape
    if rows < cols:  # Add dummy rows
        dummy_rows = np.zeros((cols - rows, cols))
        matrix = np.vstack([matrix, dummy_rows])
    elif cols < rows:  # Add dummy columns
        dummy_cols = np.zeros((rows, rows - cols))
        matrix = np.hstack([matrix, dummy_cols])
    return matrix

def reduce_matrix(matrix):
    """
    Perform row and column reductions on the cost matrix.
    """
    # Row reduction
    matrix -= np.min(matrix, axis=1).reshape(-1, 1)
    
    # Column reduction
    matrix -= np.min(matrix, axis=0)
    
    return matrix

def find_minimum_lines(matrix):
    """
    Cover all zeros in the matrix with the minimum number of lines.
    Returns the rows and columns that are covered.
    """
    covered_rows = set()
    covered_cols = set()
    zero_positions = np.argwhere(matrix == 0)

    # Iteratively cover zeros
    while zero_positions.size > 0:
        row_counts = np.sum(matrix == 0, axis=1)
        col_counts = np.sum(matrix == 0, axis=0)

        # Cover the row or column with the most zeros
        if np.max(row_counts) >= np.max(col_counts):
            row = np.argmax(row_counts)
            covered_rows.add(row)
            matrix[row, :] = np.inf  # Mark row as covered
        else:
            col = np.argmax(col_counts)
            covered_cols.add(col)
            matrix[:, col] = np.inf  # Mark column as covered

        # Update zero positions
        zero_positions = np.argwhere(matrix == 0)

    return covered_rows, covered_cols

def adjust_matrix(matrix, covered_rows, covered_cols):
    """
    Adjust the matrix when the number of covering lines is less than the size of the matrix.
    """
    # Find the smallest uncovered value
    uncovered_rows = np.setdiff1d(np.arange(matrix.shape[0]), list(covered_rows))
    uncovered_cols = np.setdiff1d(np.arange(matrix.shape[1]), list(covered_cols))
    
    if len(uncovered_rows) > 0 and len(uncovered_cols) > 0:
        uncovered = matrix[np.ix_(uncovered_rows, uncovered_cols)]
        min_value = np.min(uncovered)

        # Adjust uncovered and covered intersections
        matrix[np.ix_(uncovered_rows, uncovered_cols)] -= min_value
        matrix[np.ix_(list(covered_rows), list(covered_cols))] += min_value

    return matrix

def find_assignments(matrix, original_matrix):
    """
    Find the optimal assignment based on the final zero positions.
    Calculate the total cost using the original matrix.
    """
    assignment = []
    while np.sum(matrix == 0) > 0:
        for i in range(matrix.shape[0]):
            row_zeros = np.where(matrix[i, :] == 0)[0]
            if len(row_zeros) == 1:
                j = row_zeros[0]
                assignment.append((i, j))
                matrix[i, :] = float('inf')  # Mark row as used
                matrix[:, j] = float('inf')  # Mark column as used
                break
    # Calculate the total cost using the original matrix
    total_cost = sum(original_matrix[i, j] for i, j in assignment if i < original_matrix.shape[0] and j < original_matrix.shape[1])
    return assignment, total_cost

def hungarian_method(matrix):
    """
    Solve the assignment problem using the Hungarian Method.
    """
    print("\n--- Hungarian Method Started ---")
    
    # Save the original matrix (for calculating total cost)
    original_matrix = matrix.astype(float)
    
    # Ensure the matrix is square
    matrix = add_dummy_rows_and_columns(matrix.astype(float))
    print("\nInitial Matrix (after adding dummy variables if needed):")
    print(matrix)
    
    # Step 1: Reduce the matrix
    matrix = reduce_matrix(matrix)
    print("\nMatrix After Row and Column Reduction:")
    print(matrix)
    
    while True:
        # Step 2: Cover all zeros with minimum lines
        covered_rows, covered_cols = find_minimum_lines(matrix.copy())
        if len(covered_rows) + len(covered_cols) >= matrix.shape[0]:
            break
        # Step 3: Adjust the matrix
        matrix = adjust_matrix(matrix, covered_rows, covered_cols)
        print("\nMatrix After Adjustment:")
        print(matrix)
    
    # Step 4: Find the optimal assignment
    assignment, total_cost = find_assignments(matrix, original_matrix)
    print("\n--- Final Output ---")
    print(f"Optimal Assignment: {assignment}")
    print(f"Minimum Total Cost: {total_cost}")
    return assignment, total_cost

# User Input
def get_user_matrix():
    """
    Prompt the user to input a cost matrix.
    """
    print("Enter the dimensions of the matrix (rows x columns):")
    rows = int(input("Number of rows: "))
    cols = int(input("Number of columns: "))
    
    print("\nEnter the elements of the matrix row by row (space-separated):")
    matrix = []
    for i in range(rows):
        row = list(map(float, input(f"Row {i + 1}: ").split()))
        matrix.append(row)
    
    return np.array(matrix)

# Main Execution
print("--- Hungarian Method Solver ---")
user_matrix = get_user_matrix()
assignment, total_cost = hungarian_method(user_matrix)


--- Hungarian Method Solver ---
Enter the dimensions of the matrix (rows x columns):
Number of rows: 2
Number of columns: 3

Enter the elements of the matrix row by row (space-separated):
Row 1: 2 4 5 
Row 2: 4 5 7

--- Hungarian Method Started ---

Initial Matrix (after adding dummy variables if needed):
[[2. 4. 5.]
 [4. 5. 7.]
 [0. 0. 0.]]

Matrix After Row and Column Reduction:
[[0. 2. 3.]
 [0. 1. 3.]
 [0. 0. 0.]]

Matrix After Adjustment:
[[0. 1. 2.]
 [0. 0. 2.]
 [1. 0. 0.]]

--- Final Output ---
Optimal Assignment: [(0, 0), (1, 1), (2, 2)]
Minimum Total Cost: 7.0
